### Imports


In [ ]:
# !pip install google-genai scipy -q
# !pip install boto3 -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    precision_recall_fscore_support,
)
import re
import math
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins
import signal
import subprocess
import tempfile
import textwrap

from scipy.stats import binomtest, norm

import multiprocessing
import traceback

from typing import List, Dict, Any, Tuple, Callable, Optional, Union

In [ ]:
# Structural watermarking utilities
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../../../scripts').resolve()))

from bigram_analysis.binning_strategy import (
    load_binning_scheme,
    get_green_structural_hints,
)
from metrics.ast_bigram_detector import StructuralWatermarkDetector
from utils.structural_watermark_utils import (
    get_task_specific_green_bins,
    get_structural_embedding_prompt,
    evaluate_both_watermarks,
)

### Constants


In [ ]:
MODEL_NAME = "claude"
MODEL_PATH = "gemini-2.5-flash"

# Project root directory (absolute path to avoid confusion with relative paths)
BASE_DIR = "../../../"  # or '.'

# Derived paths
DATASET_PATH = f"../../../datasets/human_eval_164.parquet"

# Watermark parameters
Z_THRESHOLD = 2.12  # corresponds to a p-value of approximately 0.017 and confidence level of 98.3%
P_THRESHOLD = norm.sf(Z_THRESHOLD)
print(f"Using Z_THRESHOLD={Z_THRESHOLD} which corresponds to P_THRESHOLD={P_THRESHOLD:.5f}")

# GAMMA = 0.514161
SEED_KEY = "exp2025"
SMALL_SAMPLE_THRESHOLD = 30
COMMENT_ENABLED = True
ITER_CAP = 5

In [ ]:
import json
import os

# Load structural watermarking binning scheme and display seed configuration
scheme_path = "../../../data/bigram_binning_scheme_v1.json"
try:
    with open(scheme_path, "r") as f:
        binning_scheme = json.load(f)
    print(f"✅ Loaded binning scheme from {scheme_path}")
    print(f"   - Total bins: {binning_scheme['binning_config']['total_bins']}")
    print(f"   - Green bins: {len(binning_scheme['green_bins']['ids'])}")
    print(f"   - Gamma (baseline): {binning_scheme['binning_config']['gamma_baseline']:.3f}")
    
    # Display watermark seed information
    try:
        from utils.structural_watermark_utils import get_global_watermark_seed
        seed = get_global_watermark_seed()
        seed_source = 'WATERMARK_SECRET_SEED' if os.environ.get('WATERMARK_SECRET_SEED') else ('WATERMARK_SEED' if os.environ.get('WATERMARK_SEED') else 'default')
        print(f"\n🔐 Global Watermark Seed Configuration:")
        print(f"   - Current seed: {seed}")
        print(f"   - Source: {seed_source}")
        print(f"   - ✨ Watermarks are GLOBAL (same for all code, verifiable without task_id)")
    except ImportError as e:
        print(f"⚠️  Import error: {e}")
        print("   - Please restart the notebook kernel and run the imports cell again")
        print("   - Or run: import sys; sys.path.insert(0, '../../../scripts')")
except FileNotFoundError:
    print(f"⚠️  Warning: Binning scheme not found at {scheme_path}")
    binning_scheme = None

In [ ]:
missing_ids = [
    'HumanEval/24', 'HumanEval/25', 'HumanEval/26', 'HumanEval/27', 'HumanEval/28', 'HumanEval/29', 'HumanEval/30', 'HumanEval/31', 'HumanEval/33', 'HumanEval/37', 'HumanEval/38', 'HumanEval/39', 'HumanEval/40', 'HumanEval/41', 'HumanEval/42', 'HumanEval/43', 'HumanEval/44', 'HumanEval/45', 'HumanEval/46', 'HumanEval/47', 'HumanEval/48', 'HumanEval/49', 'HumanEval/50', 'HumanEval/51', 'HumanEval/52', 'HumanEval/53', 'HumanEval/54', 'HumanEval/55', 'HumanEval/56', 'HumanEval/57', 'HumanEval/58', 'HumanEval/59', 'HumanEval/60', 'HumanEval/61', 'HumanEval/62', 'HumanEval/64', 'HumanEval/65', 'HumanEval/66', 'HumanEval/67', 'HumanEval/69', 'HumanEval/70', 'HumanEval/71', 'HumanEval/72', 'HumanEval/73', 'HumanEval/75', 'HumanEval/76', 'HumanEval/77', 'HumanEval/78', 'HumanEval/79', 'HumanEval/82', 'HumanEval/83', 'HumanEval/84', 'HumanEval/86', 'HumanEval/87', 'HumanEval/91', 'HumanEval/92', 'HumanEval/93', 'HumanEval/94', 'HumanEval/95', 'HumanEval/96', 'HumanEval/97', 'HumanEval/99', 'HumanEval/100', 'HumanEval/101', 'HumanEval/102', 'HumanEval/103', 'HumanEval/104', 'HumanEval/105', 'HumanEval/106', 'HumanEval/107', 'HumanEval/108', 'HumanEval/109', 'HumanEval/110', 'HumanEval/115', 'HumanEval/116', 'HumanEval/117', 'HumanEval/118', 'HumanEval/119', 'HumanEval/120', 'HumanEval/121', 'HumanEval/122', 'HumanEval/123', 'HumanEval/124', 'HumanEval/125', 'HumanEval/127', 'HumanEval/128', 'HumanEval/129', 'HumanEval/130', 'HumanEval/132', 'HumanEval/133', 'HumanEval/134', 'HumanEval/135', 'HumanEval/136', 'HumanEval/137', 'HumanEval/138', 'HumanEval/140', 'HumanEval/141', 'HumanEval/143', 'HumanEval/144', 'HumanEval/145', 'HumanEval/146', 'HumanEval/147', 'HumanEval/149', 'HumanEval/150', 'HumanEval/151', 'HumanEval/152', 'HumanEval/153', 'HumanEval/154', 'HumanEval/155', 'HumanEval/156', 'HumanEval/157', 'HumanEval/158', 'HumanEval/159', 'HumanEval/160', 'HumanEval/161', 'HumanEval/162', 'HumanEval/163'
]

In [ ]:
df = pd.read_parquet(DATASET_PATH)
# df = df[:1]  # For testing purposes, limit to first 2 records
df = df[df['task_id'].isin(missing_ids)]
# df = df[:10]
df.shape

In [ ]:
# Experiment parameters
EXPERIMENT_NUMBER = "exp4"
EXP_VERSION = "v1-1"
GENERATION_TYPE = "during"  # 'during' or 'after'
SAMPLE_SIZE = df.shape[0]
DATASET = "humaneval"

RESULTS_CSV = f"{BASE_DIR}/results/raw/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}.csv"
OUTPUT_DIR = f"{BASE_DIR}/output/{MODEL_NAME}_{EXPERIMENT_NUMBER}_{GENERATION_TYPE}_gen_{EXP_VERSION}_{SAMPLE_SIZE}_{DATASET}"

### Adaptive Green Tokens


In [ ]:
import hashlib
import random

## Humaneval letter frequency distribution
freq_data = {
    "letter_freqs": {
        "a": 36,
        "b": 17,
        "c": 31,
        "d": 19,
        "e": 23,
        "f": 10,
        "g": 6,
        "h": 3,
        "i": 94,
        "j": 12,
        "k": 10,
        "l": 47,
        "m": 23,
        "n": 78,
        "o": 17,
        "p": 19,
        "q": 1,
        "r": 20,
        "s": 65,
        "t": 16,
        "u": 0,
        "v": 10,
        "w": 5,
        "x": 36,
        "y": 7,
        "z": 1,
    },
    "total_identifiers": 606,
}

def calculate_gamma(
    letter_counts: Dict[str, int], total_count: int, green_letters: set
) -> float:
    if total_count == 0:
        return 0.0

    gamma = 0.0
    for letter in green_letters:
        if letter in letter_counts:
            p_letter = letter_counts[letter] / total_count
            gamma += p_letter

    return gamma


def get_dynamic_tier_counts(
    G, task_id, const=SEED_KEY, wH_range=(0.40, 0.60), wM_range=(0.25, 0.40)
) -> Dict[str, float]:
    #! deterministic seed
    seed_val = int(hashlib.md5((const).encode()).hexdigest(), 16)
    rnd = random.Random(seed_val)  # local RNG so no global side effects

    # sample weights uniformly within ranges
    wH = rnd.uniform(*wH_range)
    wM = rnd.uniform(*wM_range)

    # ensure wL within bounds by clipping if necessary
    wL = 1.0 - wH - wM
    # if wL out of [0.10, 0.35], nudge wM to make it valid
    if wL < 0.10:
        # reduce wH or wM to lift wL
        deficit = 0.10 - wL
        # reduce wH first, then wM
        wH = max(wH_range[0], wH - deficit * 0.6)
        wM = max(wM_range[0], wM - deficit * 0.4)
        wL = 1.0 - wH - wM
    elif wL > 0.35:
        excess = wL - 0.35
        wH = min(wH_range[1], wH + excess * 0.6)
        wM = min(wM_range[1], wM + excess * 0.4)
        wL = 1.0 - wH - wM

    # integer counts
    gH = max(1, int(math.floor(G * wH)))
    gM = max(1, int(math.floor(G * wM)))
    gL = G - gH - gM
    # fix rounding issues
    if gL < 1:
        # push some from gH/gM to gL
        if gM > 1:
            gM -= 1
            gL += 1
        else:
            gH -= 1
            gL += 1

    return {"wH": wH, "wM": wM, "wL": wL, "gH": gH, "gM": gM, "gL": gL}


def get_red_green_sets(
    task_id, const=SEED_KEY, freq_dict=freq_data["letter_freqs"]
) -> Tuple[set, set, float]:

    task_id = str(task_id)

    # ----- Step 1: sort by frequency -----
    sorted_letters = sorted(freq_dict.keys(), key=lambda k: freq_dict[k], reverse=True)
    n = len(sorted_letters)
    high = sorted_letters[:8]
    medium = sorted_letters[8:16]
    low = sorted_letters[16:]

    # print("HIGH TIER:", high)
    # print("MEDIUM TIER:", medium)
    # print("LOW TIER:", low)

    #! ----- Step 2: unique seed -----
    seed_val = int(hashlib.md5((const).encode()).hexdigest(), 16)
    random.seed(seed_val)

    # print("RANDOM SEED:", seed_val)

    # ----- Step 3: adaptive ratio -----
    ratio = 0.5 + (random.random() - 0.5) * 0.2  # between 0.4–0.6
    green_count = int(n * ratio)

    # print(f"GREEN RATIO: {ratio:.3f}, GREEN COUNT: {green_count}/{n}")

    # ----- Step 4: weighted selection -----
    counts = get_dynamic_tier_counts(green_count, task_id, const="exp2025")
    num_high = counts["gH"]
    num_medium = counts["gM"]
    num_low = counts["gL"]
    # num_high = int(green_count * 0.5)
    # num_medium = int(green_count * 0.3)
    # num_low = green_count - num_high - num_medium

    # print("HIGH COUNT:", num_high)
    # print("MEDIUM COUNT:", num_medium)
    # print("LOW COUNT:", num_low)

    green_letters = (
        random.sample(high, min(num_high, len(high)))
        + random.sample(medium, min(num_medium, len(medium)))
        + random.sample(low, min(num_low, len(low)))
    )

    # print("INITIAL GREEN LETTERS:", green_letters)

    # Fill up if short (due to small tiers)
    while len(green_letters) < green_count:
        extra = random.choice(sorted_letters)
        if extra not in green_letters:
            green_letters.append(extra)

    # Remaining letters → red set
    red_letters = [l for l in sorted_letters if l not in green_letters]
    return set(green_letters), set(red_letters), ratio

### Helper Methods


In [ ]:
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins
import keyword

COMMON_STD_METHODS = {
    "self",
    "re",
    "cls",
    "append",
    "join",
    "dummy_function",
    "find",
    "kwargs",
    "args",
    "range",
    "print",
    "len",
    "dict",
    "list",
    "str",
    "int",
    "float",
    "set",
    "tuple",
    "os",
    "np",
    "subprocess",
    "now",
    "today",
    "timedelta",
    "strptime",
    "date",
    "time",
    "datetime",
    "logging",
    "log",
    "info",
    "debug",
    "error",
    "warning",
    "exception",
    "lower",
    "upper",
    "strip",
    "split",
    "replace",
    "endswith",
    "startswith",
    "append",
    "extend",
    "insert",
    "remove",
    "pop",
    "sort",
    "clear",
    "keys",
    "values",
    "items",
    "get",
    "update",
    "copy",
    "format",
    "join",
    "count",
    "index",
}

BUILTIN_NAMES = set(dir(builtins)).union(COMMON_STD_METHODS)


class CodeNavigator(ast.NodeVisitor):
    def __init__(self):
        self.public_classes = set()
        self.non_public_classes = set()
        self.public_funcs = set()
        self.non_public_funcs = set()
        self.public_vars = set()
        self.non_public_vars = set()
        self.docstrings = []

    def visit_FunctionDef(self, node):
        name = node.name

        # classify
        if name.startswith("__") and name.endswith("__"):
            pass
        elif name.startswith("_"):
            self.non_public_funcs.add(name)
        else:
            self.public_funcs.add(name)

        # add arguments
        for arg in node.args.args:
            if arg.arg not in BUILTIN_NAMES:
                self.non_public_vars.add(arg.arg)

        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "function_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": doc,
                }
            )
        self.generic_visit(node)

    def visit_ClassDef(self, node):
        if node.name.startswith("_"):
            self.non_public_classes.add(node.name)
        else:
            self.public_classes.add(node.name)
        self.non_public_vars.add(node.name)

        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "class_docstring",
                    "name": node.name,
                    "line_number": node.lineno,
                    "content": doc,
                }
            )
        self.generic_visit(node)

    def visit_Assign(self, node):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id not in BUILTIN_NAMES:
                if target.id.isupper():
                    self.public_vars.add(target.id)
                else:
                    self.non_public_vars.add(target.id)
        self.generic_visit(node)

    def visit_Attribute(self, node):
        # Detect method calls like self.get_possible_moves
        if isinstance(node.value, ast.Name) and node.value.id == "self":
            if node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
                # treat as a public method
                self.public_funcs.add(node.attr)
        elif node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
            # treat other attributes normally
            self.non_public_vars.add(node.attr)
        self.generic_visit(node)

    def visit_Name(self, node):
        if node.id not in BUILTIN_NAMES:
            self.non_public_vars.add(node.id)
        self.generic_visit(node)

    def visit_Module(self, node):
        doc = ast.get_docstring(node)
        if doc:
            self.docstrings.append(
                {
                    "type": "module_docstring",
                    "name": "__main__",
                    "line_number": getattr(node, "lineno", 1),
                    "content": doc,
                }
            )
        self.generic_visit(node)


def get_tokens(code):
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return set()

    visitor = CodeNavigator()
    visitor.visit(tree)

    all_tokens = (
        visitor.public_classes
        # | visitor.public_funcs
        | visitor.non_public_funcs
        | visitor.non_public_vars
        | visitor.non_public_classes
    )

    # 🚫 Remove known stdlib or logging-related names
    cleaned_tokens = {
        t for t in all_tokens if t not in COMMON_STD_METHODS and t not in BUILTIN_NAMES
    }

    return cleaned_tokens


def load_generated_code(file_path):
    if not os.path.exists(file_path):
        return None
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    return content.strip() if content.strip() else None


#! test generated output

def extract_assertions_from_check_function(check_code: str) -> List[str]:
    """Extract all assert statements from a check function."""
    try:
        tree = ast.parse(check_code)
        assertions = []
        
        for node in ast.walk(tree):
            if isinstance(node, ast.Assert):
                assertion_code = ast.unparse(node)
                assertions.append(assertion_code)
        
        return assertions
    except:
        return []

def rename_function_to_candidate(code: str) -> str:
    """Rename the first function definition to 'candidate'."""
    try:
        tree = ast.parse(code)
        
        class FunctionRenamer(ast.NodeTransformer):
            def __init__(self):
                self.renamed = False
            
            def visit_FunctionDef(self, node):
                if not self.renamed:
                    node.name = "candidate"
                    self.renamed = True
                return node
        
        renamer = FunctionRenamer()
        tree = renamer.visit(tree)
        ast.fix_missing_locations(tree)
        return ast.unparse(tree)
    except:
        return code

def run_code_with_tests(code: str, test_imports: List[str], tests: List[str], return_dict: Dict) -> Dict:
    passed, failed = 0, 0
    return_dict["error"] = ""
    return_dict["result"] = (passed, failed)
    assertions_list = []

    try:
        code = textwrap.dedent(code)
        code = rename_function_to_candidate(code)
        env = {}

        for imp in test_imports:
            exec(imp, env, env)

        exec(code, env, env)

        for test_idx, test in enumerate(tests, 1):
            try:
                exec(test, env, env)
                passed += 1
                assertions_list.append({
                    "test_num": test_idx,
                    "assertion": test,
                    "status": "PASSED"
                })
            except AssertionError as e:
                failed += 1
                error_msg = str(e) if str(e) else "Assertion failed"
                assertions_list.append({
                    "test_num": test_idx,
                    "assertion": test,
                    "status": "FAILED",
                    "message": error_msg
                })
                return_dict["error"] += f"{test}\n{error_msg}\n"
            except Exception as e:
                failed += 1
                assertions_list.append({
                    "test_num": test_idx,
                    "assertion": test,
                    "status": "ERROR",
                    "message": str(e)
                })
                return_dict["error"] += f"{test}\n{str(e)}\n"

        return_dict["result"] = (passed, failed)
        return_dict["assertions"] = str(assertions_list)

    except Exception as e:
        tb = traceback.format_exc()
        return_dict["result"] = (0, len(tests))
        return_dict["error"] = tb
        return_dict["assertions"] = str([])

    return return_dict

def safe_exec_with_tests(code: str, test_imports: List[str], tests: List[str], timeout: float = 2) -> Dict[str, Any]:
    manager = multiprocessing.Manager()
    return_dict = manager.dict()
    p = multiprocessing.Process(target=run_code_with_tests, args=(code, test_imports, tests, return_dict))
    
    p.start()
    p.join(timeout)
    
    if p.is_alive():
        p.terminate()
        return_dict['result'] = (0, len(tests))
        return_dict['error'] = "Timeout: Code execution exceeded time limit"
        return_dict['assertions'] = str([])
        return dict(return_dict)
    
    result_dict = dict(return_dict)
    
    if 'assertions' in result_dict:
        try:
            result_dict['assertions'] = eval(result_dict['assertions'])
        except:
            result_dict['assertions'] = []
    
    return result_dict

def test_code(code: str, test_imports: List[str], tests: List[str], timeout: float = 2) -> Dict:
    return safe_exec_with_tests(code, test_imports, tests, timeout)


#! Extract starting letters from comments
def extract_comments_from_source(source_code: str) -> List[Dict[str, Any]]:
    comments = []

    # create a deepcopy
    import copy

    source_code = copy.deepcopy(source_code)

    # Split into lines to process each line individually
    lines = source_code.split("\n")

    for line_number, line in enumerate(lines, 1):
        # Find all # symbols and extract comments after them
        hash_index = line.find("#")
        if hash_index != -1:
            # Extract everything after the # symbol
            comment_content = line[hash_index + 1 :].strip()
            if comment_content:  # Skip empty comments
                # Determine if it's an inline comment or full-line comment
                code_before_hash = line[:hash_index].strip()
                comment_type = (
                    "inline_comment" if code_before_hash else "full_line_comment"
                )

                comments.append(
                    {
                        "line_number": line_number,
                        "content": comment_content,
                        "type": comment_type,
                        "code_context": (
                            code_before_hash[:50] + "..."
                            if len(code_before_hash) > 50
                            else code_before_hash
                        ),
                    }
                )
    # Also extract docstrings using AST visitor
    tree = ast.parse(source_code)
    visitor = CodeNavigator()
    visitor.visit(tree)

    docstrings = visitor.docstrings

    comments.extend(docstrings)

    return comments

def get_comment_starting_letters(source_code: str, gamma) -> tuple:

    try:
        comments = extract_comments_from_source(source_code)

        # print(f"EXTRACTED COMMENTS:\n {[comment['content'] for comment in comments]}\n")

        starting_letters = []
        relevant_words = set()

        for comment in comments:
            # Split comment content by newlines to handle multi-line comments
            comment_lines = comment["content"].split("\n")

            for line in comment_lines:
                line = line.strip()
                if not line:
                    continue

                # Extract the first word from this line
                words = re.findall(r"\b[a-zA-Z]+\b", line)
                if words:
                    first_word = words[0].lower()
                    relevant_words.add(first_word)

                    # Get starting letter of the first word
                    if first_word:
                        first_char = first_word[0].lower()
                        if first_char.isalpha():
                            starting_letters.append(first_char)

        # print(f"Relevant words: {len(relevant_words)}")

        # Use global green_letters and gamma
        green_count = sum(1 for letter in starting_letters if letter in green_letters)
        token_count = len(starting_letters)

        if token_count > 0:
            p_hat = green_count / token_count
            z_score = (p_hat - gamma) / (
                (gamma * (1 - gamma)) ** 0.5 / token_count**0.5
            )
            p_value = norm.sf(z_score)  # one-tailed test
        else:
            z_score = 0.0
            p_value = 1.0

        return starting_letters, relevant_words, green_count, z_score, p_value

    except Exception as e:
        print(f"❌ Error extracting comment letters: {type(e).__name__}: {e}")
        import traceback

        traceback.print_exc()
        return [], set(), 0, 0.0, 1.0


#! fix the method name with the name mentioned in assert statement

def extract_function_names_from_code(code: str):
    """Extract all function names defined in the user code."""
    tree = ast.parse(code)
    return [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]

def extract_function_name_from_tests(test_list):
    """Extract the function name used in assert statements."""
    for test in test_list:
        try:
            tree = ast.parse(test)
            for node in ast.walk(tree):
                # Detect function call inside assert or math.isclose( func(...) )
                if isinstance(node, ast.Call):
                    # Handle nested calls like math.isclose(func(...))
                    for arg in node.args:
                        if isinstance(arg, ast.Call) and isinstance(arg.func, ast.Name):
                            return arg.func.id
                    if isinstance(node.func, ast.Name):
                        return node.func.id
        except SyntaxError:
            continue
    return None

def replace_function_name(code, old_name, new_name):
    """Safely rename the function in the code using AST."""

    class RenameTransformer(ast.NodeTransformer):
        def visit_FunctionDef(self, node):
            if node.name == old_name:
                node.name = new_name
            return node

    tree = ast.parse(code)
    tree = RenameTransformer().visit(tree)
    ast.fix_missing_locations(tree)
    return ast.unparse(tree)

def fix_method_name(user_code, test_list):
    """If needed, rename user's function to match test case function name."""
    code_func_names = extract_function_names_from_code(user_code)
    test_func_name = extract_function_name_from_tests(test_list)

    if not test_func_name:
        print("⚠️ No function found in test cases.")
        return user_code

    # Case 1: If test function name already exists in code, no change needed
    if test_func_name in code_func_names:
        return user_code

    # Case 2: Otherwise, rename the first user function to match test case
    if code_func_names:
        old_name = code_func_names[0]
        updated_code = replace_function_name(user_code, old_name, test_func_name)
        print(f"🔄 Renamed '{old_name}' → '{test_func_name}'")
        return updated_code

    print("⚠️ No function found in user code.")
    return user_code

In [ ]:
import json
from datetime import datetime

FAILURE_LOG_PATH = (
    f"{MODEL_NAME}_{EXPERIMENT_NUMBER}_{EXP_VERSION}_failures_{datetime.now().strftime('%Y%m%d_%H_%M_%S')}.json"
)


def log_failure(record_id, failure_type, error_details):
    """Append failure details into a JSON file."""
    log_entry = {
        "task_id": record_id,
        "failure_type": failure_type,
        "error_details": error_details,
        "timestamp": datetime.now().isoformat(),
    }

    try:
        # Load existing log file
        if os.path.exists(FAILURE_LOG_PATH):
            with open(FAILURE_LOG_PATH, "r") as f:
                data = json.load(f)
        else:
            data = []

        data.append(log_entry)

        with open(FAILURE_LOG_PATH, "w") as f:
            json.dump(data, f, indent=4)

    except Exception as e:
        print(f"[Warning] Failed to log error: {e}")

In [ ]:
import math
from scipy.stats import binomtest, norm


def calculate_z_score(token_count, green_count, gamma):
    """Calculate z-score for green token count (standard normal approximation)."""
    if token_count == 0 or gamma <= 0 or gamma >= 1:
        return float("nan")
    return (green_count - gamma * token_count) / math.sqrt(
        gamma * (1 - gamma) * token_count
    )


def calculate_p_value_exact(green_count, token_count, gamma):
    """
    Calculate exact binomial p-value (one-sided test: H1: p > gamma).
    Use this for ALL samples; it's exact and always valid.
    """
    if token_count == 0:
        return float("nan")
    return binomtest(green_count, token_count, gamma, alternative="greater").pvalue


def calculate_p_value_normal(z_score):
    """
    Calculate approximate p-value using standard normal CDF (one-sided).
    Used only for reference/large samples where normal approx is valid.
    """
    if math.isnan(z_score):
        return 1.0
    return norm.sf(z_score)


def get_unified_detection_score(
    token_count, green_count, gamma, small_threshold=SMALL_SAMPLE_THRESHOLD
):
    """
    UNIFIED APPROACH: Compute both z and p_exact, then select based on sample size.
    Returns: (p_unified, z_score, p_exact, method_used)

    For small samples: Use exact binomial p-value (most accurate)
    For large samples: Can use either (both converge), we use p_exact for consistency

    Score for ranking: -log10(p_unified) ensures higher scores = more evidence of watermark
    """
    z_score = calculate_z_score(token_count, green_count, gamma)
    p_exact = calculate_p_value_exact(green_count, token_count, gamma)
    p_normal = calculate_p_value_normal(z_score)

    # Select which p-value to use
    if token_count < small_threshold:
        p_unified = p_exact
        method = "binomial"
    else:
        p_unified = z_score  # ✅ Use exact for consistency; normal approx also valid
        method = "exact_large"

    # Score for ROC: higher = more evidence. Use -log10 to spread values
    score = -np.log10(np.clip(p_unified, 1e-300, 1.0))

    return {
        "p_unified": p_unified,
        "z_score": z_score,
        "p_exact": p_exact,
        "p_normal": p_normal,
        "score": score,
        "method_used": method,
        "token_count": token_count,
    }

In [ ]:
def detect_watermark(original_code, generated_code, green_letters, red_letters, gamma):
    """
    UNIFIED WATERMARK DETECTION
    - Uses consistent p-value based approach for both decisions and ROC scoring
    - Computes both exact binomial p-value and z-score
    - Decision rule: p_unified < P_THRESHOLD (one-sided)
    - Score for ROC: -log10(p_unified)
    """
    import ast

    def get_starting_letters(code):
        try:
            tree = ast.parse(code)
        except SyntaxError:
            return [], [], 0, float("nan"), float("nan"), {}

        visitor = CodeNavigator()
        visitor.visit(tree)

        # Collect non-public identifiers (user-defined variables, functions, etc.)
        non_public_tokens = (
            visitor.non_public_classes
            | visitor.non_public_funcs
            | visitor.non_public_vars
        )

        # Get starting letters, lowercase, excluding common ones like 'self', 'cls'
        relevant_tokens = [
            token for token in non_public_tokens if token not in {"self", "cls"}
        ]

        def get_starting_letter(word):
            if not word:
                return None
            if word[0] == "_":
                if len(word) > 1 and word[1].isalpha():
                    return word[1].lower()
                else:
                    return None
            elif word[0].isalpha():
                return word[0].lower()
            else:
                return None

        starting_letters = [
            letter
            for word in relevant_tokens
            if (letter := get_starting_letter(word)) is not None
        ]

        green_count = sum(1 for letter in starting_letters if letter in green_letters)

        # Include comment starting letters if enabled
        if COMMENT_ENABLED:
            cmnt_starting_letters, cmn_relevant_words, cmnt_green_count, _, _ = (
                get_comment_starting_letters(code, gamma)
            )
            # print(f"✅ COMMENT STARTING LETTERS: {cmnt_starting_letters}")
            # print(
            #     f"   Comment green count: {cmnt_green_count}, Comment token count: {len(cmnt_starting_letters)}"
            # )
            starting_letters.extend(cmnt_starting_letters)
            relevant_tokens.extend(cmn_relevant_words)
            green_count += cmnt_green_count

        token_count = len(starting_letters)

        # Get unified detection stats
        unified_stats = get_unified_detection_score(token_count, green_count, gamma)

        return starting_letters, relevant_tokens, green_count, unified_stats

    orig_starts, orig_relevant_tokens, orig_green_count, orig_stats = (
        get_starting_letters(original_code)
    )
    print("ORIGINAL TOKENS: ", orig_relevant_tokens, "LEN: ", len(orig_relevant_tokens))
    print("ORIGINAL GREEN COUNT: ", orig_green_count)
    print(
        f"ORIGINAL STATS: method={orig_stats['method_used']}, p={orig_stats['p_unified']:.6f}, z={orig_stats['z_score']:.4f}"
    )

    gen_starts, gen_relevant_tokens, gen_green_count, gen_stats = get_starting_letters(
        generated_code
    )
    print("GENERATED TOKENS: ", gen_relevant_tokens, "LEN: ", len(gen_relevant_tokens))
    print("GENERATED GREEN COUNT: ", gen_green_count)
    print(
        f"GENERATED STATS: method={gen_stats['method_used']}, p={gen_stats['p_unified']:.6f}, z={gen_stats['z_score']:.4f}"
    )

    preferred = green_letters
    avoided = red_letters

    # Calculate ratios
    orig_preferred_ratio = (
        sum(1 for letter in orig_starts if letter in preferred) / len(orig_starts)
        if orig_starts
        else 0
    )
    gen_preferred_ratio = (
        sum(1 for letter in gen_starts if letter in preferred) / len(gen_starts)
        if gen_starts
        else 0
    )

    orig_avoided_count = sum(1 for letter in orig_starts if letter in avoided)
    gen_avoided_count = sum(1 for letter in gen_starts if letter in avoided)

    # ✅ UNIFIED DETECTION: Use p_unified for both decision and all returned stats
    def is_watermarked_unified(p_unified, threshold=P_THRESHOLD):
        """Simple threshold-based decision using unified p-value."""
        if math.isnan(p_unified):
            return False
        return p_unified < threshold

    return {
        # Original code stats
        "original_z_score": orig_stats["z_score"],
        "original_p_exact": orig_stats["p_exact"],
        "original_p_unified": orig_stats["p_unified"],
        "original_score": orig_stats["score"],
        "original_method_used": orig_stats["method_used"],
        "original_preferred_ratio": orig_preferred_ratio,
        "original_token_count": len(orig_relevant_tokens),
        "original_green_count": orig_green_count,
        "original_is_watermarked": is_watermarked_unified(orig_stats["p_unified"]),
        "original_unique_starts": sorted(set(orig_starts)),
        
        # Generated code stats
        "generated_z_score": gen_stats["z_score"],
        "generated_p_exact": gen_stats["p_exact"],
        "generated_p_unified": gen_stats["p_unified"],
        "generated_score": gen_stats["score"],
        "generated_method_used": gen_stats["method_used"],
        "generated_preferred_ratio": gen_preferred_ratio,
        "generated_token_count": len(gen_relevant_tokens),
        "generated_green_count": gen_green_count,
        "generated_is_watermarked": is_watermarked_unified(gen_stats["p_unified"]),
        "generated_unique_starts": sorted(set(gen_starts)),
    }

### Watermark Embedding


#### Gemini-2.5-Flash


In [ ]:
from google import genai
from google.genai import types
import os
import re
from pathlib import Path
import ast

API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("API_KEY")

if not API_KEY:
    raise RuntimeError(
        "Missing API key. Please set GEMINI_API_KEY or API_KEY in your environment (e.g., .env)."
    )

client = genai.Client(api_key=API_KEY)

In [ ]:
# Quick sanity check: invoke Gemini with a tiny prompt
try:
    test_response = client.models.generate_content(
        model=MODEL_PATH,
        contents=[types.Content(role="user", parts=[types.Part.from_text(text="Say 'ok'. And tell me the current time in this format: YYYY-MM-DD HH:MM:SS")])],
    )
    print("✅ Gemini response:", test_response.text.strip())
except Exception as e:
    print("❌ Gemini invocation failed:", e)

#### PROMPT TEMPLATE REFERENCE:

````tex
Wang, C. Y., DaghighFarsoodeh, A., & Pham, H. V. (2024).
Selection of prompt engineering techniques for code generation through predicting code complexity.
arXiv preprint arXiv:2409.16416.```
````


In [ ]:
SYSTEM_PROMPT = """You are an expert Python programmer. Generate clean, correct code that solves the problem following the below contraints:

**OBJECTIVE #1 (PRIORITY): Code Correctness**
- Write code that is functionally correct and passes all test cases
- Verify edge cases are handled properly
- NO bugs, no off-by-one errors, complete solutions

**OBJECTIVE #2: Lexical Watermarking (Critical for acceptance)**
Your code MUST use identifiers and comments that starts with SPECIFIC starting letters:
LIST OF SPECIFIC STARTING LETTERS: {green_words}
  
Examples of AVOIDING naming patterns (DO NOT USE):
  - For green letters like 'a', 'b', 'c': a_sum, b_product, c_result, add_values, compute_max
  - For 'x', 'y', 'z': x_count, y_array, z_helper, extract_value, yield_result

**OBJECTIVE #3: Code Organization**
MUST deliver strong control flow for structural patterns:
1. **Iteration**: Use for/while loops (not list comprehensions alone)
   - Example: for item in items: process_item(item)
2. **Conditionals**: Use if-else chains (not ternary operators alone)
   - Example: if condition: do_something() else: do_other()
3. **Helper Functions**: Create and call helper functions
   - Example: define helper(), then call it from main code
4. **Nesting**: Use nested control structures where logical
   - Example: for loop containing if-else with function calls

**WRITING GUIDELINES:**
1. Prioritize correctness above all else
2. Use ONLY {green_words} starting letters when naming your identifiers (local variables, function parameters, private helper functions, internal class attributes, temporary variables) & comments.
3. Use comments where needed only, avoid using excessive comments for each statement as it breaks the naturalness of source code.
3. Organize code with natural control flow (loops, conditionals, helper functions)
4. Keep solutions concise and readable
5. Provide at most 2-3 line explanation after the code

Do not comment on naming conventions or code structure - just provide the solution.
"""

PROBLEM_TEMPLATE = (
   "Generate Python code for:\n\n{prompt}"
)

In [ ]:
# import os
# from openai import OpenAI
# from dotenv import load_dotenv

# load_dotenv()

# client = OpenAI(
#    base_url="https://openrouter.ai/api/v1",
#    api_key="sk-",
# )

In [ ]:
# def generate_code(record, feedback_prompt=""):

#     #! Get green and red letter sets
#     task_id = record["task_id"]
#     global green_letters, red_letters
#     green_letters, red_letters, ratio = get_red_green_sets(task_id)

#     problem_query = record["prompt"]
    
#     # Format system prompt with green letters (TWO CHANNELS EXPLICIT)
#     system_instruction = SYSTEM_PROMPT.format(
#         green_words=", ".join(sorted(green_letters)),  # Make it explicit
#         red_words=", ".join(sorted(red_letters))
#     )
    
#     # ✅ PRINT SYSTEM PROMPT FOR VERIFICATION
#     print("\n" + "="*80)
#     print("[SYSTEM PROMPT FOR LLM]")
#     print("="*80)
#     print(system_instruction)
#     print("="*80 + "\n")
    
#     full_prompt = PROBLEM_TEMPLATE.format(prompt=problem_query)

#     # 🔧 Add feedback if provided (Tier 2 refinement)
#     if len(feedback_prompt) > 0:
#         full_prompt = full_prompt + "\n\n[FEEDBACK FROM PREVIOUS ATTEMPT]:\n" + feedback_prompt

#     print("[USER PROMPT SENT TO LLM]:")
#     print(full_prompt)
#     print("\n")

#     contents = [
#         types.Content(
#             role="user",
#             parts=[types.Part.from_text(text=full_prompt)],
#         )
#     ]

#     response = client.models.generate_content(
#         model=MODEL_PATH,
#         contents=contents,
#         config=types.GenerateContentConfig(system_instruction=system_instruction),
#     )

#     full_text = response.text.strip()  # full reasoning + code
#     code_blocks = re.findall(r"```python(.*?)```", full_text, re.DOTALL)
#     actual_code_blocks = [block.strip() for block in code_blocks if block.strip()]

#     # Extract code & reasoning separately
#     code_text = actual_code_blocks[-1] if actual_code_blocks else ""
#     explanation_text = re.sub(
#         r"```python.*?```", "", full_text, flags=re.DOTALL
#     ).strip()

#     return {
#         "code": code_text,
#         "explanation": explanation_text,
#         "full_response": full_text,
#     }

In [ ]:
import boto3

def generate_response(system_instruction: str, user_prompt: str, max_tokens: int = 2048, temperature: float = 0.1, track_tokens: bool = False):
    """Generate response from Claude via AWS Bedrock with separate system and user prompts."""
    client = boto3.client(
        'bedrock-runtime',
        region_name=os.getenv('AWS_REGION', 'us-east-1'),
        aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
        aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY')
    )
    
    model_id = os.getenv('DEFAULT_MODEL', 'us.anthropic.claude-sonnet-4-20250514-v1:0')
    
    # Combine system and user prompts
    full_message = f"{system_instruction}\n\n{user_prompt}"
    
    request_body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": full_message}]
    }
    
    response = client.invoke_model(
        modelId=model_id,
        body=json.dumps(request_body),
        contentType='application/json'
    )
    
    response_body = json.loads(response['body'].read())
    text = ""
    if 'content' in response_body and len(response_body['content']) > 0:
        text = response_body['content'][0]['text']
    
    if track_tokens:
        usage = response_body.get('usage', {})
        return {
            'text': text,
            'input_tokens': usage.get('input_tokens', 0),
            'output_tokens': usage.get('output_tokens', 0),
            'reasoning_tokens': usage.get('reasoning_tokens', 0),  # Claude may provide reasoning tokens
            'total_tokens': usage.get('input_tokens', 0) + usage.get('output_tokens', 0) + usage.get('reasoning_tokens', 0)
        }
    else:
        return text


In [ ]:
def generate_code(record, feedback_prompt=""):
    """Generate code with dual-channel watermarking using Claude via Bedrock."""
    global green_letters, red_letters
    
    task_id = record["task_id"]
    green_letters, red_letters, ratio = get_red_green_sets(task_id)

    # 📝 Generate examples of compliant (green) and non-compliant (red) identifiers
    green_list = sorted(list(green_letters))
    red_list = sorted(list(red_letters))[:8]
    
    # Create multiple examples for green letters
    green_examples = []
    for g in green_list[:6]:
        green_examples.append(f"{g}_value, {g}_count, compute_{g}, find_{g}")
    
    problem_query = record["prompt"]
    system_instruction = SYSTEM_PROMPT.format(
        green_words=", ".join(green_list),  # Full list of green letters
    )
    full_prompt = PROBLEM_TEMPLATE.format(prompt=problem_query)

    # 🔧 Add targeted feedback if provided (Multi-tier refinement)
    if len(feedback_prompt) > 0:
        full_prompt = full_prompt + "\n\n" + feedback_prompt

    # 🔍 DEBUG OUTPUT: Print prompts before sending to LLM
    print("\n" + "="*80)
    print(f"[{task_id}] LLM INVOCATION DEBUG (Claude via Bedrock)")
    print("="*80)
    # print("\n📋 SYSTEM INSTRUCTION (Watermark Rules):")
    # print("-" * 80)
    # print(system_instruction)
    # print("\n" + "-" * 80)
    # print("\n❓ USER QUERY:")
    # print("-" * 80)
    # print(full_prompt)
    # print("\n" + "-" * 80)
    print("\n⏳ Sending to Claude (Claude-Sonnet-4)...\n")

    result = generate_response(system_instruction, full_prompt, max_tokens=2048, track_tokens=True)
    
    full_text = result['text'].strip()
    input_tokens = result['input_tokens']
    output_tokens = result['output_tokens']
    reasoning_tokens = result.get('reasoning_tokens', 0)
    
    code_blocks = re.findall(r"```python(.*?)```", full_text, re.DOTALL)
    actual_code_blocks = [block.strip() for block in code_blocks if block.strip()]
    code_text = actual_code_blocks[-1] if actual_code_blocks else ""
    explanation_text = re.sub(r"```python.*?```", "", full_text, flags=re.DOTALL).strip()

    return {
        "code": code_text,
        "explanation": explanation_text,
        "full_response": full_text,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": reasoning_tokens,
        "total_tokens": input_tokens + output_tokens + reasoning_tokens
    }

In [ ]:
def evaluate_candidate(record, code):
    """
    Compute correctness and watermark fidelity (both lexical and structural).
    ✅ DUAL-CHANNEL WATERMARKING: Uses exact binomial test for lexical + structural detection.
    """

    #! Get green/red letter sets for this task
    task_id = record["task_id"]

    letter_freqs = freq_data["letter_freqs"]
    total_identifiers = freq_data["total_identifiers"]

    global green_letters, red_letters
    green_letters, red_letters, ratio = get_red_green_sets(task_id)
    GAMMA = calculate_gamma(letter_freqs, total_identifiers, green_letters)

    # Correctness via unit tests
    test_str = record.get("test", [])
    assertions = extract_assertions_from_check_function(test_str)
    result = test_code(code, [], assertions)

    passed, failed = result["result"]
    error_message = result["error"]
    correctness = (failed == 0) and (passed > 0)

    # ==== LEXICAL WATERMARK DETECTION ====
    original_code = record.get("canonical_solution", "") or ""
    original_code = textwrap.dedent(original_code)
    detection_result = (
        detect_watermark(original_code, code, green_letters, red_letters, GAMMA)
        if code
        else {}
    )

    z = detection_result.get("generated_z_score", None)
    p_unified = detection_result.get("generated_p_unified", None)
    p_exact = detection_result.get("generated_p_exact", None)
    score = detection_result.get("generated_score", float("nan"))
    method_used = detection_result.get("generated_method_used", "unknown")
    token_count = detection_result.get("generated_token_count", 0)
    green_count = detection_result.get("generated_green_count", 0)
    green_tokens = detection_result.get("generated_unique_starts", [])
    meets_z = bool(detection_result.get("generated_is_watermarked", False))

    # ==== STRUCTURAL WATERMARK DETECTION ====
    structural_result = {}
    structural_detected = False
    structural_z = None
    structural_p = None
    structural_token_count = 0
    structural_green_count = 0

    if binning_scheme and code:
        try:
            detector = StructuralWatermarkDetector(scheme_path, verbose=False)
            structural_result = detector.detect(code, task_id=task_id)
            structural_detected = structural_result.get("watermark_detected", False)
            structural_z = structural_result.get("z_score", None)
            structural_p = structural_result.get("p_value", None)
            structural_token_count = structural_result.get("total_recognized", 0)
            structural_green_count = structural_result.get("green_count", 0)
        except Exception as e:
            print(f"⚠️  Error in structural detection: {e}")

    return {
        # Correctness
        "correctness": bool(correctness),
        "error_message": error_message if len(error_message) > 0 else None,
        "passed": int(passed),
        "failed": int(failed),
        # Lexical watermark
        "z": z,
        "p_unified": p_unified,
        "p_exact": p_exact,
        "score": score,
        "method_used": method_used,
        "token_count": int(token_count) if isinstance(token_count, (int, float)) else 0,
        "green_count": int(green_count) if isinstance(green_count, (int, float)) else 0,
        "meets_z": bool(meets_z),
        "watermark_failed": not meets_z,
        "watermark_details": (
            (z, p_unified, token_count, green_count, green_tokens)
            if not meets_z
            else None
        ),
        # Structural watermark
        "structural_z": float(structural_z) if structural_z is not None else None,
        "structural_p": float(structural_p) if structural_p is not None else None,
        "structural_detected": bool(structural_detected),
        "structural_token_count": int(structural_token_count),
        "structural_green_count": int(structural_green_count),
        # Combined
        "both_detected": bool(meets_z and structural_detected),
        "either_detected": bool(meets_z or structural_detected),
    }

In [ ]:
def generate_constraint_status_summary(correct, has_lexical, has_structural, green_count, token_count, lex_z, struct_z, dual_channel=True):
    """Generate a clear status summary showing which constraints are met/not met."""
    green_pct = (green_count / token_count * 100) if token_count > 0 else 0
    
    status_marks = []
    met_count = 0
    unmet_count = 0
    
    # Correctness
    if correct:
        status_marks.append("✅ CORRECTNESS: Passed all unit tests")
        met_count += 1
    else:
        status_marks.append("❌ CORRECTNESS: Failed unit tests")
        unmet_count += 1
    
    # Lexical watermark
    if has_lexical:
        status_marks.append(f"✅ LEXICAL: Detected (z={lex_z:.2f}, {green_pct:.0f}% green)")
        met_count += 1
    else:
        status_marks.append(f"❌ LEXICAL: NOT detected (z={lex_z:.2f}, {green_pct:.0f}% green, need >70%)")
        unmet_count += 1
    
    # Structural watermark
    if dual_channel:
        if has_structural:
            status_marks.append(f"✅ STRUCTURAL: Detected (z={struct_z:.2f})")
            met_count += 1
        else:
            status_marks.append(f"❌ STRUCTURAL: NOT detected (z={struct_z:.2f})")
            unmet_count += 1
    
    return {
        "summary": "\n".join(status_marks),
        "met_count": met_count,
        "unmet_count": unmet_count,
        "green_pct": green_pct,
    }


def generate_targeted_feedback(correct, has_lexical, has_structural, eval_res, green_letters, red_letters, dual_channel=True):
    """Generate highly specific feedback based on which constraints failed."""
    feedback_parts = []
    
    # Include overall constraint status
    green_count = eval_res.get("green_count", 0)
    token_count = eval_res.get("token_count", 0)
    lex_z = eval_res.get("z", float('nan'))
    struct_z = eval_res.get("structural_z", float('nan'))
    
    status = generate_constraint_status_summary(correct, has_lexical, has_structural, green_count, token_count, lex_z, struct_z, dual_channel)
    feedback_parts.append("[CONSTRAINT STATUS]")
    feedback_parts.append(status["summary"])
    feedback_parts.append("")
    
    # === Constraint 1: Correctness ===
    if not correct:
        error_msg = eval_res.get("error_message", "Unknown error")
        feedback_parts.append("[CORRECTNESS FIX REQUIRED]")
        feedback_parts.append(f"Error: {error_msg[:200]}")
        feedback_parts.append("→ Debug and fix the error before proceeding to other constraints.")
        feedback_parts.append("")
    else:
        feedback_parts.append("[✅ CORRECTNESS CONSTRAINT MET]")
        feedback_parts.append("")
    
    # === Constraint 2: Lexical Watermark ===
    if not has_lexical:
        green_sample = sorted(list(green_letters))[:8]
        green_pct = (green_count / token_count * 100) if token_count > 0 else 0
        feedback_parts.append("[LEXICAL WATERMARK FIX REQUIRED]")
        feedback_parts.append(f"Current: {green_count}/{token_count} identifiers ({green_pct:.0f}%) start with green letters")
        feedback_parts.append(f"Required: {green_sample} (and only these)")
        feedback_parts.append(f"Target: ≥70% of identifiers MUST start with green letters")
        feedback_parts.append("")
        feedback_parts.append("ACTION: Rename ALL variables, functions, and helper methods to use ONLY these green starting letters:")
        feedback_parts.append(f"{', '.join(green_sample)}")
        feedback_parts.append("")
        feedback_parts.append("EXAMPLES of compliant names:")
        examples = [f"{g}_result, {g}_count, {g}_helper, compute_{g}, find_{g}, calculate_{g}" for g in green_sample[:4]]
        feedback_parts.append(" | ".join(examples))
        feedback_parts.append("")
        feedback_parts.append("CRITICAL: Every identifier you create must start with ONE of these green letters. No exceptions.")
        feedback_parts.append("")
    else:
        feedback_parts.append("[✅ LEXICAL WATERMARK CONSTRAINT MET]")
        feedback_parts.append("")
    
    # === Constraint 3: Structural Watermark ===
    if dual_channel:
        if not has_structural:
            feedback_parts.append("[STRUCTURAL WATERMARK FIX REQUIRED]")
            feedback_parts.append(f"Current z-score: {struct_z:.2f} (need z > {Z_THRESHOLD:.2f})")
            feedback_parts.append("")
            feedback_parts.append("ACTION: Strengthen control flow with:")
            feedback_parts.append("  1. Use for/while loops for iteration (not list comprehensions alone)")
            feedback_parts.append("  2. Create helper functions and call them from main code")
            feedback_parts.append("  3. Use if-else chains for conditional logic (not ternary operators alone)")
            feedback_parts.append("  4. Add nested control structures where possible")
            feedback_parts.append("")
            feedback_parts.append("EXAMPLE pattern:")
            feedback_parts.append("  for item in items:")
            feedback_parts.append("      result = helper_function(item)  # Call helper")
            feedback_parts.append("      if result > threshold:")
            feedback_parts.append("          output.append(result)")
            feedback_parts.append("")
        else:
            feedback_parts.append("[✅ STRUCTURAL WATERMARK CONSTRAINT MET]")
            feedback_parts.append("")
    
    return "\n".join(feedback_parts)


def run_phase1(
    record, iter_cap=ITER_CAP, verbose=True, dual_channel=True
):
    """
    Phase 1: MULTI-ITERATION DUAL-CHANNEL WATERMARKING (up to ITER_CAP iterations)
    
    🎯 Goal: All three constraints must succeed together:
      1. ✅ CORRECTNESS: Code passes all unit tests
      2. ✅ LEXICAL: Identifier naming with green letters (>70% green)
      3. ✅ STRUCTURAL: Control flow patterns for AST watermarking
    
    ⚙️ ALGORITHM:
      For each iteration (up to ITER_CAP):
        1. Generate code
        2. Evaluate: Check all constraints → get detailed status
        3. If ALL three constraints pass → STOP. Success!
        4. If ANY constraint fails → Generate targeted feedback:
           - Show which constraints passed/failed
           - Provide specific guidance for each failed constraint
           - Emphasize the lexical watermark requirement if that's the blocker
        5. Retry with feedback
      
      Continue until all constraints met OR ITER_CAP reached.
      
    📊 Output: Best attempt with detailed constraint tracking
    """
    best = None
    selected = None
    all_iterations = []
    feedback_prompt = ""

    for it in range(1, iter_cap + 1):
        if verbose:
            print(f"\n[{record['task_id']}] ITERATION {it}/{iter_cap}: ", end="", flush=True)

        # Generate code with feedback from previous iteration
        gen = generate_code(record, feedback_prompt)
        code = gen["code"]
        reasoning_text = gen["explanation"]

        if verbose:
            print(f"Generated → Evaluating...", flush=True)

        # Evaluate all three constraints
        eval_res = evaluate_candidate(record, code)
        eval_res.update(
            {
                "iteration": it,
                "reasoning_text": reasoning_text,
                "full_llm_response": gen["full_response"],
                "code": code,
                "input_tokens": gen.get("input_tokens", 0),
                "output_tokens": gen.get("output_tokens", 0),
                "reasoning_tokens": gen.get("reasoning_tokens", 0),
                "total_tokens": gen.get("total_tokens", 0),
            }
        )
        
        all_iterations.append(eval_res)

        # Extract constraint states
        correct = eval_res["correctness"]
        has_lexical = eval_res["meets_z"]
        has_structural = eval_res.get("structural_detected", False) if dual_channel else True
        
        green_count = eval_res.get("green_count", 0)
        token_count = eval_res.get("token_count", 0)
        lex_z = eval_res.get("z", float('nan'))
        struct_z = eval_res.get("structural_z", float('nan'))
        
        # Check if all constraints met
        all_constraints_met = correct and has_lexical and has_structural
        
        # Display constraint status
        if verbose:
            status = generate_constraint_status_summary(correct, has_lexical, has_structural, green_count, token_count, lex_z, struct_z, dual_channel)
            print(f"  Met: {status['met_count']}/{3 if dual_channel else 2} constraints")
            print(f"  {status['summary']}")
        
        if all_constraints_met:
            if verbose:
                print(f"\n  🎉 SUCCESS at iteration {it} - All constraints met!")
            selected = eval_res
            break
        
        # Not all constraints met - check if we should continue
        if it >= iter_cap:
            if verbose:
                print(f"\n  ⚠️ Iteration {it}/{iter_cap} - Reached ITER_CAP, accepting best attempt")
            selected = eval_res
            break
        
        # Generate targeted feedback for next iteration
        green_letters, red_letters, _ = get_red_green_sets(record["task_id"])
        feedback_prompt = generate_targeted_feedback(correct, has_lexical, has_structural, eval_res, green_letters, red_letters, dual_channel)
        
        if verbose:
            print(f"\n  📝 Generated targeted feedback for iteration {it + 1}...")

    if selected is None and all_iterations:
        selected = all_iterations[-1]
    
    if verbose and selected:
        print(f"\n[{record['task_id']}] Final result: Iteration {selected.get('iteration')}")

    return selected

### RUN

In [ ]:
import time

def process_dataset(df, output_dir, dual_channel=True, max_iters=ITER_CAP):
    Path(output_dir).parent.mkdir(exist_ok=True)
    results = []

    for idx, record in df.iterrows():
        task_id = record.get("task_id") or record.get("id")
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)

        try:
            sel = run_phase1(
                record, iter_cap=max_iters, verbose=True, dual_channel=dual_channel
            )
            code = sel.get("code", "") if sel else ""

            # Save code
            output_file = out_dir / f"{task_id.split('/')[1]}.py"
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(code or "")

            # Persist result row with both watermark channels
            results.append(
                {
                    "task_id": task_id,
                    "iteration_used": sel.get("iteration"),
                    "correct": sel.get("correctness"),
                    "tests_passed": sel.get("passed"),
                    "tests_failed": sel.get("failed"),
                    # Lexical watermark
                    "z_score": sel.get("z"),
                    "p_unified": sel.get("p_unified"),
                    "p_exact": sel.get("p_exact"),
                    "score": sel.get("score"),
                    "method_used": sel.get("method_used"),
                    "token_count": sel.get("token_count"),
                    "green_count": sel.get("green_count"),
                    "meets_z": sel.get("meets_z"),
                    # Structural watermark
                    "structural_z_score": sel.get("structural_z"),
                    "structural_p_value": sel.get("structural_p"),
                    "structural_token_count": sel.get("structural_token_count"),
                    "structural_green_count": sel.get("structural_green_count"),
                    "structural_detected": sel.get("structural_detected"),
                    # Combined
                    "both_detected": sel.get("both_detected"),
                    "either_detected": sel.get("either_detected"),
                    # Token usage
                    "input_tokens": sel.get("input_tokens", 0),
                    "output_tokens": sel.get("output_tokens", 0),
                    "reasoning_tokens": sel.get("reasoning_tokens", 0),
                    "total_tokens": sel.get("total_tokens", 0),
                    # Execution
                    "stopped_early": sel.get("stopping_condition_met"),
                    "output_file": str(output_file),
                }
            )
        except Exception as e:
            print(f"[Phase1] ❌ Failure for {task_id}: {e}")
            import traceback
            traceback.print_exc()
            results.append(
                {
                    "task_id": task_id,
                    "error": str(e),
                    "iteration_used": None,
                    "correct": False,
                    "tests_passed": 0,
                    "tests_failed": 0,
                    "z_score": float("nan"),
                    "p_unified": float("nan"),
                    "p_exact": float("nan"),
                    "score": float("nan"),
                    "method_used": "error",
                    "token_count": 0,
                    "green_count": 0,
                    "meets_z": False,
                    "structural_z_score": float("nan"),
                    "structural_p_value": float("nan"),
                    "structural_token_count": 0,
                    "structural_green_count": 0,
                    "structural_detected": False,
                    "both_detected": False,
                    "either_detected": False,
                    "input_tokens": 0,
                    "output_tokens": 0,
                    "reasoning_tokens": 0,
                    "total_tokens": 0,
                    "stopped_early": False,
                    "output_file": None,
                }
            )

        print(f"\n{idx+1}/{len(df)} processed: {task_id.split('/')[1]}\n")

    results_df = pd.DataFrame(results)
    Path(RESULTS_CSV).parent.mkdir(parents=True, exist_ok=True)

    results_df.to_csv(RESULTS_CSV, index=False)
    print(f"✅ Results saved to {RESULTS_CSV}")
    
    # Print summary statistics
    print("\n" + "="*80)
    print("FINAL SUMMARY STATISTICS")
    print("="*80)
    print(f"Total tasks processed: {len(results_df)}")
    print(f"✅ Correctness: {results_df['correct'].sum()}/{len(results_df)} ({results_df['correct'].mean()*100:.1f}%)")
    print(f"✅ Lexical watermark: {results_df['meets_z'].sum()}/{len(results_df)} ({results_df['meets_z'].mean()*100:.1f}%)")
    if 'structural_detected' in results_df.columns:
        print(f"✅ Structural watermark: {results_df['structural_detected'].sum()}/{len(results_df)} ({results_df['structural_detected'].mean()*100:.1f}%)")
    if 'both_detected' in results_df.columns:
        print(f"✅ BOTH watermarks: {results_df['both_detected'].sum()}/{len(results_df)} ({results_df['both_detected'].mean()*100:.1f}%)")
    print(f"📊 Avg iterations used: {results_df['iteration_used'].mean():.1f}")
    
    # Token usage statistics
    if 'total_tokens' in results_df.columns:
        valid_tokens = results_df['total_tokens'].dropna()
        if len(valid_tokens) > 0:
            print(f"🔢 Token Usage:")
            print(f"  • Input tokens: {results_df['input_tokens'].sum():,}")
            print(f"  • Output tokens: {results_df['output_tokens'].sum():,}")
            print(f"  • Reasoning tokens: {results_df['reasoning_tokens'].sum():,}")
            print(f"  • Total tokens: {results_df['total_tokens'].sum():,}")
            print(f"  • Avg total tokens per task: {results_df['total_tokens'].mean():.0f}")
    
    print("="*80 + "\n")

    return results

In [ ]:
# Test with explicit dual-channel prompts on 1 task (HumanEval/24)
df_test = df[:10]  # Run only 10 tasks to verify both watermarks
print(f"Testing on: {df_test['task_id'].values[0]}")
df_test['task_id'].head()

In [ ]:
# Run with explicit dual-channel watermark enforcement
results_test = process_dataset(df_test, OUTPUT_DIR, dual_channel=True)

In [ ]:
df_test.head()

### Comparing Actual vs Watermarked Sample

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap
from pathlib import Path

print("=" * 80)
print("🔍 WATERMARK DETECTION COMPARISON: Actual vs Watermarked AI")
print("=" * 80)

project_root = Path(BASE_DIR).resolve()
dataset_path = project_root / "datasets" / "human_eval_164.parquet"
watermarked_dir = project_root / "output" / "claude_exp4_during_gen_v1-1_117_humaneval"

print(f"\n📂 HumanEval dataset: {dataset_path}")
print(f"📂 Watermarked solutions: {watermarked_dir}")

df_human_eval = pd.read_parquet(dataset_path)
print(f"\n📦 Loaded {len(df_human_eval)} records from human_eval_164.parquet")

sample_limit = 10
comparison_entries = []
processed = 0


def gather_detection(record, code, label):
    normalized_code = textwrap.dedent(code or "")
    try:
        eval_result = evaluate_candidate(record, normalized_code)
    except Exception as exc:
        print(f"  ⚠️  Detection failed for {record.get('task_id')} ({label}): {exc}")
        return None

    token_count = max(int(eval_result.get("token_count", 0)), 0)
    green_count = max(int(eval_result.get("green_count", 0)), 0)
    struct_tokens = max(int(eval_result.get("structural_token_count", 0)), 0)
    struct_green = max(int(eval_result.get("structural_green_count", 0)), 0)

    lex_z = eval_result.get("z")
    struct_z = eval_result.get("structural_z")

    entry = {
        "source": label,
        "task_id": record.get("task_id"),
        "task_num": record.get("task_id", "").split("/")[-1],
        "lex_z": float(lex_z) if lex_z is not None else np.nan,
        "lex_p": eval_result.get("p_unified"),
        "lex_detected": bool(eval_result.get("meets_z")),
        "lex_tokens": token_count,
        "lex_green": green_count,
        "lex_green_pct": (green_count / token_count * 100) if token_count > 0 else 0.0,
        "struct_z": float(struct_z) if struct_z is not None else np.nan,
        "struct_p": eval_result.get("structural_p"),
        "struct_tokens": struct_tokens,
        "struct_green": struct_green,
        "struct_green_pct": (struct_green / struct_tokens * 100) if struct_tokens > 0 else 0.0,
        "struct_detected": bool(eval_result.get("structural_detected")),
        "correctness": bool(eval_result.get("correctness")),
    }
    return entry

print(f"\n⏳ Running detection pipeline on up to {sample_limit} tasks that have watermarked outputs...")

for _, record in df_human_eval.iterrows():
    if processed >= sample_limit:
        break

    task_id = record.get("task_id") or record.get("id")
    if not task_id:
        continue
    task_num = task_id.split("/")[-1]
    watermarked_file = watermarked_dir / f"{task_num}.py"

    if not watermarked_file.exists():
        continue

    canonical_code = record.get("canonical_solution", "")
    if not canonical_code.strip():
        continue

    try:
        watermarked_code = watermarked_file.read_text(encoding="utf-8")
    except Exception:
        print(f"  ⚠️  Could not read watermarked file for task {task_id}")
        continue

    actual_entry = gather_detection(record, canonical_code, "Actual")
    watermarked_entry = gather_detection(record, watermarked_code, "Watermarked")

    if actual_entry and watermarked_entry:
        comparison_entries.extend([actual_entry, watermarked_entry])
        processed += 1
        print(f"   ✓ Processed task {task_num} ({processed}/{sample_limit})")

if processed == 0:
    raise RuntimeError("No tasks with both actual and watermarked solutions were processed.")

comparison_df = pd.DataFrame(comparison_entries)
print(f"\n✅ Completed detection on {processed} paired tasks ({len(comparison_df)} evaluation rows)")

agg_metrics = (
    comparison_df.groupby("source", as_index=False)
    .agg(
        tasks=("task_id", "nunique"),
        lex_detection_rate=("lex_detected", "mean"),
        struct_detection_rate=("struct_detected", "mean"),
        lex_z_mean=("lex_z", "mean"),
        lex_z_std=("lex_z", "std"),
        struct_z_mean=("struct_z", "mean"),
        struct_z_std=("struct_z", "std"),
        lex_tokens_mean=("lex_tokens", "mean"),
        lex_green_pct=("lex_green_pct", "mean"),
        struct_tokens_mean=("struct_tokens", "mean"),
        struct_green_pct=("struct_green_pct", "mean"),
        correctness_rate=("correctness", "mean"),
    )
)

token_stats = agg_metrics[["source", "tasks", "lex_tokens_mean", "lex_green_pct", "lex_detection_rate"]].copy()
token_stats.columns = ["Source", "Task Count", "Avg Lex Tokens", "Avg Green %", "Lexical Detection Rate"]
token_stats["Lexical Detection Rate"] = token_stats["Lexical Detection Rate"] * 100

struct_stats = agg_metrics[["source", "struct_tokens_mean", "struct_green_pct", "struct_detection_rate", "struct_z_mean", "struct_z_std"]].copy()
struct_stats.columns = ["Source", "Avg Struct Tokens", "Avg Green %", "Structural Detection Rate", "Mean Struct Z", "Struct Z Std"]
struct_stats["Structural Detection Rate"] = struct_stats["Structural Detection Rate"] * 100

print("\n📊 Table 1: Token Statistics (Lexical Focus)")
print(token_stats.round(2).to_string(index=False))

print("\n📊 Table 2: Structural Statistics")
print(struct_stats.round(2).to_string(index=False))

# print("\n📈 Detection Comparison Plots")
# fig, axes = plt.subplots(1, 2, figsize=(16, 5))
# fig.suptitle("Watermark Detection: Actual vs Watermarked AI", fontsize=14, fontweight="bold")

# x = np.arange(len(agg_metrics))
# width = 0.35

# axes[0].bar(x - width / 2, agg_metrics["lex_detection_rate"] * 100, width, label="Lexical")
# axes[0].bar(x + width / 2, agg_metrics["struct_detection_rate"] * 100, width, label="Structural")
# axes[0].set_xticks(x)
# axes[0].set_xticklabels(agg_metrics["source"])
# axes[0].set_ylabel("Detection Rate (%)")
# axes[0].set_title("Detection Rates by Source")
# axes[0].legend()
# axes[0].grid(axis="y", alpha=0.3)

# axes[1].bar(x - width / 2, agg_metrics["lex_z_mean"].fillna(0), width, label="Lexical Z")
# axes[1].bar(x + width / 2, agg_metrics["struct_z_mean"].fillna(0), width, label="Structural Z")
# axes[1].set_xticks(x)
# axes[1].set_xticklabels(agg_metrics["source"])
# axes[1].set_ylabel("Mean Z-Score")
# axes[1].set_title("Average Z-Scores by Source")
# axes[1].legend()
# axes[1].grid(axis="y", alpha=0.3)

# plt.tight_layout(rect=[0, 0.03, 1, 0.95])
# plt.show()

agg_index = agg_metrics.set_index("source")
lex_pct = (agg_index["lex_detection_rate"] * 100).to_dict()
struct_pct = (agg_index["struct_detection_rate"] * 100).to_dict()
correct_pct = (agg_index["correctness_rate"] * 100).to_dict()

print("\n🧾 Final Verdict")
print(f"  • Lexical watermark constraints: Actual {lex_pct.get('Actual', 0):.1f}% vs Watermarked {lex_pct.get('Watermarked', 0):.1f}%")
print(f"  • Structural watermark constraints: Actual {struct_pct.get('Actual', 0):.1f}% vs Watermarked {struct_pct.get('Watermarked', 0):.1f}%")
print(f"  • Correctness rate: Actual {correct_pct.get('Actual', 0):.1f}% vs Watermarked {correct_pct.get('Watermarked', 0):.1f}%")
